# 90 Colab Main Pipeline

Status: Canonical convenience surface | Descriptive

This notebook wraps the current Lumis-1 active path into one sequential Colab workflow for operators who cannot run multiple notebooks side by side.

It does all of the following in order:

1. mounts Drive and bootstraps the rehab branch
2. validates the canonical identity dataset
3. builds the open corpus
4. merges and validates the final dataset
5. runs SFT
6. runs DPO
7. exports GGUF first
8. runs the current eval/export smoke surface
9. copies artifacts to Drive and optionally downloads one GGUF file to the browser

Proven vs unproven:

- the canonical identity pack is real and remains the strongest completed artifact
- the open/full dataset path is only proven after this notebook writes fresh reports and outputs
- SFT, DPO, eval, and export are only proven when `workspace/runs/<run_id>/` contains populated evidence

Dependency policy:

- by default this notebook uses the repo's pinned baseline, not blind "latest"
- the repo currently pins `unsloth==2026.2.0` and `unsloth_zoo==2026.2.0`
- official Unsloth docs still support `save_pretrained_gguf`, `save_pretrained_merged`, and an auto-installer fallback when notebook environments drift


In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("LUMIS1_REPO_URL", "https://github.com/dttdrv/lumis-1.git")
REPO_BRANCH = os.environ.get("LUMIS1_REPO_BRANCH") or "main"
REPO_DIR = Path("/content/lumis-1")

DRIVE_ROOT = Path("/content/drive/MyDrive/lumis1_colab")
IDENTITY_INPUT_DIR = DRIVE_ROOT / "identity_input"
WORKSPACE_PERSIST_ROOT = DRIVE_ROOT / "workspace"
EXPORT_PERSIST_ROOT = DRIVE_ROOT / "exports"

INSTALL_MODE = "repo_pinned"  # repo_pinned | unsloth_auto
STRICT_REPO_PINNED_SYNC = True
SOURCE_MODE = "hf"  # hf | local
STREAMING = True
DRY_RUN = False
ALLOW_SMALL_SAMPLE = False
ALLOW_UNVERIFIED_MULTIMODAL_SFT = False

PIPELINE_PREFIX = "colab-main-001"
PROFILE = "default_96gb"  # default_96gb | safe_fallback
FIRST_50_STEPS_SANITY = False

RUN_SFT = True
RUN_DPO = True
RUN_EXPORT = True
RUN_EVAL = True

GGUF_QUANTIZATION_METHODS = ["q4_k_m", "q8_0"]
DOWNLOAD_GGUF_TO_BROWSER = False
DOWNLOAD_GGUF_VARIANT_TOKEN = "q4_k_m"
EVAL_DO_SAMPLE = True
EVAL_TEMPERATURE = 0.7
EVAL_TOP_P = 0.8
EVAL_TOP_K = 20
EVAL_SEED = 3407
HF_TOKEN = None

def run(cmd: list[str], *, cwd: Path | None = None) -> None:
    rendered = " ".join(cmd)
    print(f"+ {rendered}")
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

def ensure_repo_checkout() -> Path:
    if not (REPO_DIR / "configs" / "paths.yaml").exists():
        if REPO_DIR.exists():
            shutil.rmtree(REPO_DIR)
        run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])
    else:
        run(["git", "fetch", "origin"], cwd=REPO_DIR)
        run(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR)
        run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=REPO_DIR)
    os.chdir(REPO_DIR)
    return REPO_DIR.resolve()

def replace_with_symlink(target: Path, source: Path) -> None:
    source.mkdir(parents=True, exist_ok=True)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_symlink():
        if target.resolve() == source.resolve():
            return
        target.unlink()
    elif target.exists():
        backup = target.with_name(target.name + "_repo_backup")
        if backup.exists():
            if backup.is_dir():
                shutil.rmtree(backup)
            else:
                backup.unlink()
        shutil.move(str(target), str(backup))
    target.symlink_to(source, target_is_directory=True)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive", force_remount=False)
    try:
        from google.colab import userdata  # type: ignore

        if HF_TOKEN is None:
            HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

REPO_ROOT = ensure_repo_checkout()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from lumis1.identity_pack import DEFAULT_PREFERENCE_NAMES, DEFAULT_SFT_NAMES
from lumis1.main_pipeline import build_main_colab_run_plan


def identity_files_present(path: Path) -> bool:
    return any((path / name).exists() for name in DEFAULT_SFT_NAMES) and any(
        (path / name).exists() for name in DEFAULT_PREFERENCE_NAMES
    )

replace_with_symlink(REPO_ROOT / "workspace", WORKSPACE_PERSIST_ROOT)

identity_repo_target = REPO_ROOT / "Dataset" / "identity_dataset" / "output" / "full_run_codex_spark_xhigh"
if not identity_files_present(identity_repo_target):
    if not identity_files_present(IDENTITY_INPUT_DIR):
        raise FileNotFoundError(
            "IDENTITY_INPUT_DIR must contain one accepted SFT file and one accepted preference file. "
            f"Accepted SFT names: {DEFAULT_SFT_NAMES}; accepted preference names: {DEFAULT_PREFERENCE_NAMES}; checked: {IDENTITY_INPUT_DIR}"
        )
    replace_with_symlink(identity_repo_target, IDENTITY_INPUT_DIR)

RUN_PLAN = build_main_colab_run_plan(REPO_ROOT, PIPELINE_PREFIX)
SFT_RUN_ID = str(RUN_PLAN["sft_run_id"])
DPO_RUN_ID = str(RUN_PLAN["dpo_run_id"])
EXPORT_RUN_ID = str(RUN_PLAN["export_run_id"])
EVAL_RUN_ID = str(RUN_PLAN["eval_run_id"])

print("repo:", REPO_ROOT)
print("identity:", identity_repo_target)
print("workspace:", REPO_ROOT / "workspace")
print("run ids:", SFT_RUN_ID, DPO_RUN_ID, EXPORT_RUN_ID, EVAL_RUN_ID)


In [ ]:
from __future__ import annotations

import importlib
import importlib.metadata
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
REQUIRED_MODULES = {
    "yaml": "pyyaml",
    "datasets": "datasets",
    "transformers": "transformers",
    "trl": "trl",
    "accelerate": "accelerate",
    "peft": "peft",
    "unsloth": "unsloth",
    "unsloth_zoo": "unsloth_zoo",
    "PIL": "Pillow",
    "jsonschema": "jsonschema",
    "langdetect": "langdetect",
    "bitsandbytes": "bitsandbytes",
}

def read_exact_pins() -> dict[str, str]:
    pins: dict[str, str] = {}
    for raw_line in (REPO_ROOT / "constraints.txt").read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if "==" in line:
            name, version = line.split("==", maxsplit=1)
            pins[name.strip().lower()] = version.strip()
    return pins

def installed_version(package_name: str) -> str | None:
    try:
        return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        return None

missing = [
    package_name
    for module_name, package_name in REQUIRED_MODULES.items()
    if importlib.util.find_spec(module_name) is None
]

sync_reasons: list[str] = []
if INSTALL_MODE == "repo_pinned" and STRICT_REPO_PINNED_SYNC:
    exact_pins = read_exact_pins()
    for package_name, expected in sorted(exact_pins.items()):
        current = installed_version(package_name)
        if current != expected:
            sync_reasons.append(f"{package_name}={current} expected {expected}")
    transformers_version = installed_version("transformers")
    if transformers_version is None or not transformers_version.startswith("5."):
        sync_reasons.append(f"transformers={transformers_version} expected major 5")

if not missing and not sync_reasons:
    print({"install_mode": INSTALL_MODE, "missing_modules": [], "sync_reasons": [], "action": "none"})
else:
    print(
        {
            "install_mode": INSTALL_MODE,
            "missing_modules": missing,
            "sync_reasons": sync_reasons,
            "action": "install_then_restart",
        }
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip"], check=True)
    if INSTALL_MODE == "repo_pinned":
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-c",
                "constraints.txt",
                "-r",
                "requirements.txt",
            ],
            cwd=str(REPO_ROOT),
            check=True,
        )
    elif INSTALL_MODE == "unsloth_auto":
        subprocess.run(
            "wget -qO- https://raw.githubusercontent.com/unslothai/unsloth/main/unsloth/_auto_install.py | python -",
            cwd=str(REPO_ROOT),
            shell=True,
            check=True,
        )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-c", "constraints.txt", "-r", "requirements.txt"],
            cwd=str(REPO_ROOT),
            check=True,
        )
    else:
        raise ValueError(f"unsupported INSTALL_MODE: {INSTALL_MODE}")

    print("dependencies installed. restarting the runtime now. re-run the notebook from the top after reconnect.")
    os.kill(os.getpid(), 9)


In [ ]:
from __future__ import annotations

import importlib
import importlib.metadata
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone

import accelerate
import datasets
import peft
import torch
import transformers
import trl
import unsloth
import unsloth_zoo
from transformers import AutoTokenizer

from lumis1.identity_pack import build_identity_validation_report

REPORTS = REPO_ROOT / "workspace" / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

env_report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "trl": trl.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
    "peft": peft.__version__,
    "unsloth": getattr(unsloth, "__version__", "unknown"),
    "unsloth_zoo": getattr(unsloth_zoo, "__version__", "unknown"),
}
major = int(str(transformers.__version__).split(".", maxsplit=1)[0])
if major != 5:
    raise RuntimeError(
        f"transformers major version must be 5 for this repo baseline, got {transformers.__version__}"
    )

train_cfg = json.loads((REPO_ROOT / "workspace" / "reports" / "train_sft_config_resolved.json").read_text(encoding="utf-8")) if (REPO_ROOT / "workspace" / "reports" / "train_sft_config_resolved.json").exists() else None
base_model_name = None
if train_cfg is None:
    import yaml
    base_model_name = yaml.safe_load((REPO_ROOT / "configs" / "train_sft.yaml").read_text(encoding="utf-8"))["model"]["base_model"]
else:
    base_model_name = train_cfg["model"]["base_model"]

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
probe_messages = [{"role": "user", "content": "Say hello in one short sentence."}]
try:
    probe_render = tokenizer.apply_chat_template(
        probe_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
except TypeError as exc:
    raise RuntimeError(
        "The tokenizer chat template did not accept enable_thinking=False. "
        "Qwen3.5 non-thinking mode is not safely configured."
    ) from exc
if "<think>" in probe_render or "</think>" in probe_render:
    raise RuntimeError("The tokenizer chat template probe still emitted thinking markers.")
env_report["non_thinking_chat_template_probe"] = {
    "base_model": base_model_name,
    "passed": True,
    "render_preview": probe_render[:200],
}

freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(REPORTS / "env_sanity.json").write_text(json.dumps(env_report, indent=2), encoding="utf-8")
(REPORTS / "env_freeze.txt").write_text(freeze, encoding="utf-8")

identity_report = build_identity_validation_report(REPO_ROOT)
(REPORTS / "identity_validation.json").write_text(json.dumps(identity_report, indent=2), encoding="utf-8")

print(json.dumps({"env": env_report, "identity": identity_report}, indent=2))


In [ ]:
from __future__ import annotations

import json

from huggingface_hub import login

from lumis1.identity_pack import build_identity_validation_report

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

REPORTS = REPO_ROOT / "workspace" / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

identity_report = build_identity_validation_report(REPO_ROOT)
if identity_report["counts"]["sft_rows"] != 100000:
    raise RuntimeError(f"identity SFT count mismatch: {identity_report['counts']['sft_rows']}")
if identity_report["counts"]["preference_rows"] != 25000:
    raise RuntimeError(
        f"identity preference count mismatch: {identity_report['counts']['preference_rows']}"
    )

report_path = REPORTS / "identity_validation.json"
report_path.write_text(json.dumps(identity_report, indent=2), encoding="utf-8")

print(json.dumps(identity_report, indent=2))
print("saved:", report_path)


In [ ]:
from __future__ import annotations

import json
import uuid
from collections import Counter

import yaml

from lumis1.filters import apply_row_filters
from lumis1.hf_ingest import IngestError, load_allowlist, stream_source_records
from lumis1.license_ledger import validate_allowlist_sources
from lumis1.mixing_math import estimate_row_tokens
from lumis1.schema import validate_preference_row, validate_sft_row

REPORTS = REPO_ROOT / "workspace" / "reports"
INTERIM = REPO_ROOT / "workspace" / "interim"
REPORTS.mkdir(parents=True, exist_ok=True)
INTERIM.mkdir(parents=True, exist_ok=True)

allowlist_path = REPO_ROOT / "configs" / "dataset_sources_allowlist.yaml"
mixture_path = REPO_ROOT / "configs" / "dataset_mixture.yaml"
identity_report_path = REPORTS / "identity_validation.json"

allowlist_raw = yaml.safe_load(allowlist_path.read_text(encoding="utf-8")) or {}
mixture = yaml.safe_load(mixture_path.read_text(encoding="utf-8")) or {}
sources = [s for s in allowlist_raw.get("sources", []) if isinstance(s, dict)]
validate_allowlist_sources(sources)
allowlist_map = load_allowlist(allowlist_path)

mixture_sources = [s for s in mixture.get("sources", []) if isinstance(s, dict)]
mixture_source_map = {str(s.get("source_id")): s for s in mixture_sources if s.get("source_id")}

if not identity_report_path.exists():
    raise FileNotFoundError("Missing workspace/reports/identity_validation.json. The identity cell must run first.")
identity_report = json.loads(identity_report_path.read_text(encoding="utf-8"))
identity_tokens = int(identity_report.get("tokens", {}).get("sft_tokens_total", 0))
if identity_tokens <= 0:
    raise RuntimeError("identity token count missing or zero in identity_validation.json")

identity_share = float(mixture.get("identity_pack", {}).get("fixed_share_of_final_sft_tokens", 0.20))
open_token_budget_target = int(identity_tokens * ((1.0 - identity_share) / identity_share))
dry_run_budget = int(mixture.get("token_budgets", {}).get("open_sft_budget", {}).get("dry_run_tokens", 50000))
if DRY_RUN:
    open_token_budget_target = min(open_token_budget_target, dry_run_budget)

scan_limit_default = int(mixture.get("ingestion_defaults", {}).get("max_records_scanned_per_source", 2000000))
if DRY_RUN:
    scan_limit_default = min(scan_limit_default, 20000)

def canonical_messages(rec: dict[str, object]) -> list[dict[str, object]]:
    if isinstance(rec.get("messages"), list) and rec["messages"]:
        return list(rec["messages"])
    prompt = rec.get("prompt") or rec.get("instruction") or rec.get("question") or rec.get("input")
    answer = rec.get("response") or rec.get("output") or rec.get("answer") or rec.get("chosen")
    if not isinstance(prompt, str) or not prompt.strip():
        prompt = "No prompt provided"
    if not isinstance(answer, str) or not answer.strip():
        answer = "No response provided"
    return [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": answer},
    ]

def extract_preference_triplet(rec: dict[str, object]) -> tuple[str, str, str] | None:
    prompt = rec.get("prompt") or rec.get("instruction") or rec.get("question") or rec.get("input")
    chosen = rec.get("chosen")
    rejected = rec.get("rejected")
    if isinstance(prompt, str) and isinstance(chosen, str) and isinstance(rejected, str):
        return prompt, chosen, rejected

    a = rec.get("response_a") or rec.get("answer_a") or rec.get("candidate_a")
    b = rec.get("response_b") or rec.get("answer_b") or rec.get("candidate_b")
    label = rec.get("label") or rec.get("preference") or rec.get("winner")
    if isinstance(prompt, str) and isinstance(a, str) and isinstance(b, str) and isinstance(label, str):
        upper = label.strip().upper()
        if upper in {"A", "LEFT", "1", "CHOICE_A"}:
            return prompt, a, b
        if upper in {"B", "RIGHT", "2", "CHOICE_B"}:
            return prompt, b, a
    return None

open_sft: list[dict[str, object]] = []
open_prefs: list[dict[str, object]] = []
source_reports: list[dict[str, object]] = []
global_open_tokens = 0
global_remaining = open_token_budget_target

for source_id, source_cfg in mixture_source_map.items():
    if global_remaining <= 0:
        break

    allow_entry = allowlist_map.get(source_id)
    if not isinstance(allow_entry, dict):
        source_reports.append({"source_id": source_id, "error": "source missing in allowlist"})
        continue
    if allow_entry.get("enabled") is not True:
        source_reports.append({"source_id": source_id, "skipped": "disabled_in_allowlist"})
        continue
    if allow_entry.get("source_mode") != SOURCE_MODE:
        source_reports.append({"source_id": source_id, "skipped": "source_mode_mismatch"})
        continue

    source_budget = int(source_cfg.get("token_budget") or allow_entry.get("max_tokens_cap") or 0)
    if source_budget <= 0:
        source_reports.append({"source_id": source_id, "skipped": "token_budget_zero"})
        continue

    source_remaining = min(source_budget, global_remaining)
    source_kept_tokens = 0
    source_kept_rows = 0
    raw_rows_scanned = 0
    pref_kept_rows = 0
    drop_reasons = Counter()

    source_entry = dict(allow_entry)
    source_entry["split"] = str(allow_entry.get("default_split") or "train")

    try:
        stream = stream_source_records(
            source_entry,
            source_mode=SOURCE_MODE,
            allowlist=allowlist_map,
            limit=scan_limit_default,
            streaming=STREAMING,
        )
    except IngestError as exc:
        source_reports.append({"source_id": source_id, "error": str(exc)})
        continue

    for rec in stream:
        if source_remaining <= 0 or global_remaining <= 0:
            break

        raw_rows_scanned += 1
        base = {
            "schema_version": "1.0",
            "id": str(rec.get("id") or uuid.uuid4()),
            "source_id": source_id,
            "license": str(allow_entry.get("license", "")),
            "thinking": "off",
            "chat_template_kwargs": {"enable_thinking": False},
            "category": str(source_cfg.get("category") or "utility_tasks"),
            "modality": str(source_cfg.get("modality") or "text"),
            "messages": canonical_messages(rec),
        }

        filtered_rows, filt_report = apply_row_filters([base], drop_on_cot=True)
        for reason, count in (filt_report.get("drop_reasons") or {}).items():
            drop_reasons[reason] += int(count)
        if not filtered_rows:
            continue

        try:
            row = validate_sft_row(filtered_rows[0])
        except Exception as exc:
            drop_reasons[f"schema:{str(exc).splitlines()[0][:120]}"] += 1
            continue

        row_tokens = estimate_row_tokens(row)
        if row_tokens <= 0:
            drop_reasons["zero_token_row"] += 1
            continue
        if row_tokens > source_remaining:
            drop_reasons["source_token_budget_skip"] += 1
            continue
        if row_tokens > global_remaining:
            drop_reasons["global_token_budget_skip"] += 1
            continue

        open_sft.append(row)
        source_kept_tokens += row_tokens
        source_kept_rows += 1
        source_remaining -= row_tokens
        global_remaining -= row_tokens
        global_open_tokens += row_tokens

        pref_triplet = extract_preference_triplet(rec)
        if pref_triplet and source_id in set(mixture.get("preferences", {}).get("enabled_sources", [])):
            prompt, chosen, rejected = pref_triplet
            pref_row = {
                "id": str(rec.get("id") or uuid.uuid4()),
                "source_id": source_id,
                "license": str(allow_entry.get("license", "")),
                "thinking": "off",
                "chat_template_kwargs": {"enable_thinking": False},
                "prompt": prompt,
                "chosen": chosen,
                "rejected": rejected,
            }
            try:
                open_prefs.append(validate_preference_row(pref_row))
                pref_kept_rows += 1
            except Exception:
                drop_reasons["pref_schema_invalid"] += 1

    source_reports.append(
        {
            "source_id": source_id,
            "token_budget": source_budget,
            "kept_tokens": source_kept_tokens,
            "raw_rows_scanned": raw_rows_scanned,
            "kept_rows": source_kept_rows,
            "kept_preference_rows": pref_kept_rows,
            "drop_reasons": dict(drop_reasons),
            "source_budget_fully_used": source_remaining == 0,
        }
    )

if global_open_tokens != open_token_budget_target:
    raise RuntimeError(
        f"open token budget mismatch: expected {open_token_budget_target}, got {global_open_tokens}"
    )

open_sft_path = INTERIM / "open_sft.jsonl"
open_pref_path = INTERIM / "open_preferences.jsonl"
with open_sft_path.open("w", encoding="utf-8") as handle:
    for row in open_sft:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
with open_pref_path.open("w", encoding="utf-8") as handle:
    for row in open_prefs:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

report = {
    "source_mode": SOURCE_MODE,
    "dry_run": DRY_RUN,
    "streaming": STREAMING,
    "open_sft_token_budget_target": open_token_budget_target,
    "open_sft_tokens_built": global_open_tokens,
    "sources": source_reports,
    "totals": {
        "open_sft_rows": len(open_sft),
        "open_preference_rows": len(open_prefs),
    },
    "outputs": {
        "open_sft": str(open_sft_path),
        "open_preferences": str(open_pref_path),
    },
}

report_path = REPORTS / "open_corpus_build_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
print("saved:", report_path)


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import yaml

from lumis1.full_dataset import (
    build_dataset_manifest,
    build_full_dataset_validation_report,
    infer_modality,
    select_open_sft_rows,
)
from lumis1.identity_pack import iter_jsonl, normalize_identity_preference_row, resolve_identity_paths

REPORTS = REPO_ROOT / "workspace" / "reports"
FINAL = REPO_ROOT / "workspace" / "final"
INTERIM = REPO_ROOT / "workspace" / "interim"
REPORTS.mkdir(parents=True, exist_ok=True)
FINAL.mkdir(parents=True, exist_ok=True)

mixture = yaml.safe_load((REPO_ROOT / "configs" / "dataset_mixture.yaml").read_text(encoding="utf-8"))
identity_report = json.loads((REPORTS / "identity_validation.json").read_text(encoding="utf-8"))
identity_paths = resolve_identity_paths(REPO_ROOT)
identity_sft_path = Path(identity_paths["sft"])
identity_pref_path = Path(identity_paths["preferences"])
open_sft_path = INTERIM / "open_sft.jsonl"
open_pref_path = INTERIM / "open_preferences.jsonl"

for path in (identity_sft_path, identity_pref_path, open_sft_path, open_pref_path):
    if not path.exists():
        raise FileNotFoundError(f"missing required input for merge: {path}")

identity_sft = []
for row in iter_jsonl(identity_sft_path):
    item = dict(row)
    item.setdefault("category", "identity_behavior")
    item["modality"] = infer_modality(item)
    identity_sft.append(item)

open_sft_all = []
for row in iter_jsonl(open_sft_path):
    item = dict(row)
    item["modality"] = infer_modality(item)
    open_sft_all.append(item)

identity_pref = [
    normalize_identity_preference_row(row, idx)
    for idx, row in enumerate(iter_jsonl(identity_pref_path), start=1)
]
open_pref = list(iter_jsonl(open_pref_path))
full_pref = identity_pref + open_pref

selection = select_open_sft_rows(
    identity_rows=identity_sft,
    open_rows=open_sft_all,
    identity_share_target=float(mixture["identity_pack"].get("fixed_share_of_final_sft_tokens", 0.20)),
    allow_small_sample=ALLOW_SMALL_SAMPLE,
)
selected_open = selection["selected_open_rows"]
full_sft = identity_sft + selected_open

full_sft_path = FINAL / "full_sft.jsonl"
full_pref_path = FINAL / "full_preferences.jsonl"
with full_sft_path.open("w", encoding="utf-8") as handle:
    for row in full_sft:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
with full_pref_path.open("w", encoding="utf-8") as handle:
    for row in full_pref:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

validation_report = build_full_dataset_validation_report(
    REPO_ROOT,
    full_sft_path=full_sft_path,
    full_preferences_path=full_pref_path,
    identity_validation_report=identity_report,
    allow_small_sample=ALLOW_SMALL_SAMPLE,
)
if not validation_report["pass"]:
    raise RuntimeError(
        "full dataset validation failed:\n" + json.dumps(validation_report["validations"], indent=2)
    )

validation_report_path = REPORTS / "full_dataset_validation.json"
validation_report_path.write_text(json.dumps(validation_report, indent=2), encoding="utf-8")

mixture_math = {
    "counts": validation_report["counts"],
    "shares": validation_report["shares"],
    "targets": validation_report["targets"],
    "selection": {
        "mode": selection["selection_mode"],
        "required_open_tokens_exact": selection["required_open_tokens_exact"],
        "open_tokens_selected": selection["open_tokens"],
        "remaining_open_tokens": selection["remaining_open_tokens"],
    },
}
(REPORTS / "mixture_math.json").write_text(json.dumps(mixture_math, indent=2), encoding="utf-8")

manifest = build_dataset_manifest(
    full_sft_path=full_sft_path,
    full_preferences_path=full_pref_path,
    validation_report=validation_report,
    validation_report_path=validation_report_path,
)
(FINAL / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps({"selection": mixture_math["selection"], "validation": validation_report}, indent=2))
print("saved:", FINAL / "dataset_manifest.json")


In [ ]:
from __future__ import annotations

import json
import platform
import sys

import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

from lumis1.hashing import sha256_file
from lumis1.main_pipeline import resolve_sft_runtime
from lumis1.run_evidence import create_run_evidence_tree, write_run_status, write_run_summary

REPORTS = REPO_ROOT / "workspace" / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

sft_resolved = resolve_sft_runtime(
    REPO_ROOT,
    run_plan={
        "sft_run_id": SFT_RUN_ID,
        "sft_output_dir": REPO_ROOT / "workspace" / "runs" / SFT_RUN_ID / "artifacts" / "sft_model",
        "sft_checkpoint_dir": REPO_ROOT / "workspace" / "runs" / SFT_RUN_ID / "artifacts" / "sft_model" / "checkpoints",
    },
    profile_name=PROFILE,
    run_training=RUN_SFT,
    first_50_steps_sanity=FIRST_50_STEPS_SANITY,
)
SFT_OUTPUT_DIR = Path(sft_resolved["output_dir"])
(REPORTS / "train_sft_config_resolved.json").write_text(json.dumps(sft_resolved, indent=2), encoding="utf-8")

validation_report = json.loads((REPORTS / "full_dataset_validation.json").read_text(encoding="utf-8"))
image_text_rows = int(validation_report.get("histograms", {}).get("modality", {}).get("image_text", 0))
if image_text_rows > 0 and not ALLOW_UNVERIFIED_MULTIMODAL_SFT:
    raise RuntimeError(
        "The merged SFT dataset contains multimodal rows, and this notebook is not yet proof-bearing for "
        "Qwen3.5 vision fine-tuning. Official Unsloth guidance points to the explicit FastVisionModel path for VLM work. "
        "Refuse to continue with the unverified language-only trainer path. "
        "If you deliberately want to test the current unverified path, set ALLOW_UNVERIFIED_MULTIMODAL_SFT = True."
    )

if not RUN_SFT:
    print("RUN_SFT is False. Training skipped.")
else:
    run_paths = create_run_evidence_tree(REPO_ROOT, SFT_RUN_ID)
    (run_paths["config_snapshot"] / "train_sft_resolved.json").write_text(
        json.dumps(sft_resolved, indent=2),
        encoding="utf-8",
    )
    (run_paths["commands"] / "notebook_90_sft_invocation.txt").write_text(
        "Sequential Colab notebook SFT execution\n"
        f"profile={PROFILE}\n"
        f"output_dir={SFT_OUTPUT_DIR}\n"
        f"image_text_rows={image_text_rows}\n",
        encoding="utf-8",
    )
    (run_paths["environment"] / "runtime.json").write_text(
        json.dumps(
            {
                "python": sys.version,
                "platform": platform.platform(),
                "cwd": str(REPO_ROOT),
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    write_run_status(
        REPO_ROOT,
        SFT_RUN_ID,
        stage="sft",
        status="running",
        details={"profile": PROFILE, "output_dir": str(SFT_OUTPUT_DIR), "image_text_rows": image_text_rows},
    )

    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=sft_resolved["model"]["base_model"],
            max_seq_length=int(sft_resolved["training"].get("max_seq_length", 4096)),
            load_in_4bit=False,
            dtype=torch.bfloat16,
            trust_remote_code=True,
        )
        probe = tokenizer.apply_chat_template(
            [{"role": "user", "content": "Say hello in one short sentence."}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        if "<think>" in probe or "</think>" in probe:
            raise RuntimeError("Qwen3.5 non-thinking probe still emitted thinking markers during SFT setup.")
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"

        model = FastLanguageModel.get_peft_model(
            model,
            r=int(sft_resolved["lora"]["r"]),
            lora_alpha=int(sft_resolved["lora"]["lora_alpha"]),
            lora_dropout=float(sft_resolved["lora"]["lora_dropout"]),
            target_modules=sft_resolved["lora"]["target_modules"],
            bias=sft_resolved["lora"]["bias"],
        )

        raw_dataset = load_dataset("json", data_files=sft_resolved["dataset_path"], split="train")
        dataset_mode = "conversational_messages"
        trainer_kwargs = {}
        train_dataset = raw_dataset
        if image_text_rows == 0:
            def render_text(example: dict[str, object]) -> dict[str, str]:
                messages = example.get("messages")
                if not isinstance(messages, list):
                    raise RuntimeError("SFT row is missing messages")
                text = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=False,
                    enable_thinking=False,
                )
                if "<think>" in text or "</think>" in text:
                    raise RuntimeError("Rendered SFT text still contains thinking markers")
                return {"text": text}

            train_dataset = raw_dataset.map(render_text, remove_columns=raw_dataset.column_names)
            dataset_mode = "text_chat_template_rendered"
            trainer_kwargs["dataset_text_field"] = "text"

        trainer = SFTTrainer(
            model=model,
            processing_class=tokenizer,
            train_dataset=train_dataset,
            args=SFTConfig(**sft_resolved["training"], output_dir=str(SFT_OUTPUT_DIR)),
            **trainer_kwargs,
        )
        train_result = trainer.train()
        trainer.save_model(str(SFT_OUTPUT_DIR))
        tokenizer.save_pretrained(str(SFT_OUTPUT_DIR))

        train_metrics = getattr(train_result, "metrics", {})
        checksums = {
            str(path.relative_to(SFT_OUTPUT_DIR)): sha256_file(path)
            for path in sorted(SFT_OUTPUT_DIR.rglob("*"))
            if path.is_file()
        }
        report = {
            "run_id": SFT_RUN_ID,
            "output_dir": str(SFT_OUTPUT_DIR),
            "metrics": train_metrics,
            "artifact_files": sorted(checksums),
            "dataset_mode": dataset_mode,
            "image_text_rows": image_text_rows,
        }
        (run_paths["reports"] / "sft_training.json").write_text(
            json.dumps(report, indent=2),
            encoding="utf-8",
        )
        (run_paths["checksums"] / "artifacts.json").write_text(
            json.dumps(checksums, indent=2),
            encoding="utf-8",
        )
        write_run_status(REPO_ROOT, SFT_RUN_ID, stage="sft", status="completed", details=report)
        write_run_summary(
            REPO_ROOT,
            SFT_RUN_ID,
            "# SFT Summary\n\n```json\n" + json.dumps(report, indent=2) + "\n```",
        )
        print(json.dumps(report, indent=2))
    except Exception as exc:
        write_run_status(
            REPO_ROOT,
            SFT_RUN_ID,
            stage="sft",
            status="failed",
            details={"error": str(exc), "output_dir": str(SFT_OUTPUT_DIR), "image_text_rows": image_text_rows},
        )
        raise


In [ ]:
from __future__ import annotations

import json
import platform
import sys

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer

from lumis1.hashing import sha256_file
from lumis1.main_pipeline import resolve_dpo_runtime
from lumis1.run_evidence import create_run_evidence_tree, write_run_status, write_run_summary

REPORTS = REPO_ROOT / "workspace" / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

dpo_resolved = resolve_dpo_runtime(
    REPO_ROOT,
    run_plan={
        "dpo_run_id": DPO_RUN_ID,
        "dpo_output_dir": REPO_ROOT / "workspace" / "runs" / DPO_RUN_ID / "artifacts" / "dpo_model",
        "sft_output_dir": SFT_OUTPUT_DIR,
    },
    profile_name=PROFILE,
    run_training=RUN_DPO,
)
DPO_OUTPUT_DIR = Path(dpo_resolved["output_dir"])
(REPORTS / "train_dpo_config_resolved.json").write_text(json.dumps(dpo_resolved, indent=2), encoding="utf-8")

if not RUN_DPO:
    print("RUN_DPO is False. DPO skipped.")
else:
    if not SFT_OUTPUT_DIR.exists():
        raise FileNotFoundError(f"SFT output dir missing: {SFT_OUTPUT_DIR}")

    run_paths = create_run_evidence_tree(REPO_ROOT, DPO_RUN_ID)
    (run_paths["config_snapshot"] / "train_dpo_resolved.json").write_text(
        json.dumps(dpo_resolved, indent=2),
        encoding="utf-8",
    )
    (run_paths["commands"] / "notebook_90_dpo_invocation.txt").write_text(
        "Sequential Colab notebook DPO execution\n"
        f"profile={PROFILE}\n"
        f"output_dir={DPO_OUTPUT_DIR}\n"
        f"sft_input={SFT_OUTPUT_DIR}\n",
        encoding="utf-8",
    )
    (run_paths["environment"] / "runtime.json").write_text(
        json.dumps(
            {
                "python": sys.version,
                "platform": platform.platform(),
                "cwd": str(REPO_ROOT),
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    write_run_status(
        REPO_ROOT,
        DPO_RUN_ID,
        stage="dpo",
        status="running",
        details={"profile": PROFILE, "output_dir": str(DPO_OUTPUT_DIR)},
    )

    pref_path = REPO_ROOT / "workspace" / "final" / "full_preferences.jsonl"
    dataset = load_dataset("json", data_files=str(pref_path), split="train")

    try:
        model = AutoModelForCausalLM.from_pretrained(str(SFT_OUTPUT_DIR), torch_dtype=torch.bfloat16, trust_remote_code=True)
        tokenizer = AutoTokenizer.from_pretrained(str(SFT_OUTPUT_DIR), trust_remote_code=True)
        probe = tokenizer.apply_chat_template(
            [{"role": "user", "content": "Say hello in one short sentence."}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        if "<think>" in probe or "</think>" in probe:
            raise RuntimeError("Qwen3.5 non-thinking probe still emitted thinking markers during DPO setup.")
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"

        def format_prompt(example: dict[str, object]) -> dict[str, object]:
            prompt = example.get("prompt")
            if not isinstance(prompt, str) or not prompt.strip():
                prompt_messages = example.get("prompt_messages")
                if isinstance(prompt_messages, list):
                    prompt_parts = [
                        str(item.get("content", "")).strip()
                        for item in prompt_messages
                        if isinstance(item, dict)
                        and item.get("role") == "user"
                        and isinstance(item.get("content"), str)
                        and str(item.get("content", "")).strip()
                    ]
                    prompt = " ".join(prompt_parts).strip()
            if not isinstance(prompt, str) or not prompt.strip():
                raise RuntimeError("Preference row missing prompt or prompt_messages")
            rendered_prompt = tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
            if "<think>" in rendered_prompt or "</think>" in rendered_prompt:
                raise RuntimeError("Rendered DPO prompt still contains thinking markers")
            return {"prompt": rendered_prompt}

        dataset = dataset.map(format_prompt)
        trainer = DPOTrainer(
            model=model,
            ref_model=None,
            processing_class=tokenizer,
            train_dataset=dataset,
            args=DPOConfig(**dpo_resolved["training"], output_dir=str(DPO_OUTPUT_DIR)),
            beta=float(dpo_resolved["dpo"]["beta"]),
        )
        train_result = trainer.train()
        trainer.save_model(str(DPO_OUTPUT_DIR))
        tokenizer.save_pretrained(str(DPO_OUTPUT_DIR))

        train_metrics = getattr(train_result, "metrics", {})
        checksums = {
            str(path.relative_to(DPO_OUTPUT_DIR)): sha256_file(path)
            for path in sorted(DPO_OUTPUT_DIR.rglob("*"))
            if path.is_file()
        }
        report = {
            "run_id": DPO_RUN_ID,
            "output_dir": str(DPO_OUTPUT_DIR),
            "metrics": train_metrics,
            "artifact_files": sorted(checksums),
        }
        (run_paths["reports"] / "dpo_training.json").write_text(
            json.dumps(report, indent=2),
            encoding="utf-8",
        )
        (run_paths["checksums"] / "artifacts.json").write_text(
            json.dumps(checksums, indent=2),
            encoding="utf-8",
        )
        write_run_status(REPO_ROOT, DPO_RUN_ID, stage="dpo", status="completed", details=report)
        write_run_summary(
            REPO_ROOT,
            DPO_RUN_ID,
            "# DPO Summary\n\n```json\n" + json.dumps(report, indent=2) + "\n```",
        )
        print(json.dumps(report, indent=2))
    except Exception as exc:
        write_run_status(
            REPO_ROOT,
            DPO_RUN_ID,
            stage="dpo",
            status="failed",
            details={"error": str(exc), "output_dir": str(DPO_OUTPUT_DIR)},
        )
        raise


In [ ]:
from __future__ import annotations

import json
import platform
import sys

import torch
from peft import AutoPeftModelForCausalLM
from unsloth import FastLanguageModel, FastVisionModel

from lumis1.export_smoke import detect_gguf_files, validate_required_variants
from lumis1.hashing import sha256_file
from lumis1.main_pipeline import build_gguf_export_plan
from lumis1.run_evidence import create_run_evidence_tree, write_run_status, write_run_summary

FINAL_MODEL_DIR = DPO_OUTPUT_DIR if RUN_DPO else SFT_OUTPUT_DIR
GGUF_EXPORT_REPORT = {"status": "skipped", "reason": "RUN_EXPORT is False"}
GGUF_DIR = REPO_ROOT / "workspace" / "runs" / EXPORT_RUN_ID / "artifacts" / "gguf"

if RUN_EXPORT:
    if not FINAL_MODEL_DIR.exists():
        raise FileNotFoundError(f"final model dir missing for export: {FINAL_MODEL_DIR}")

    run_paths = create_run_evidence_tree(REPO_ROOT, EXPORT_RUN_ID)
    export_plan = build_gguf_export_plan(
        model_dir=FINAL_MODEL_DIR,
        export_dir=run_paths["artifacts"] / "gguf",
        quantization_methods=GGUF_QUANTIZATION_METHODS,
    )
    MERGED_DIR = run_paths["artifacts"] / "merged_16bit"
    GGUF_DIR = export_plan["export_dir"]
    GGUF_DIR.mkdir(parents=True, exist_ok=True)
    MERGED_DIR.mkdir(parents=True, exist_ok=True)

    (run_paths["config_snapshot"] / "export_inputs.json").write_text(
        json.dumps(
            {
                "run_id": EXPORT_RUN_ID,
                "final_model_dir": str(FINAL_MODEL_DIR),
                "gguf_quantization_methods": GGUF_QUANTIZATION_METHODS,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    (run_paths["commands"] / "notebook_90_export_invocation.txt").write_text(
        "Sequential Colab notebook GGUF export execution\n"
        f"final_model_dir={FINAL_MODEL_DIR}\n"
        f"gguf_dir={GGUF_DIR}\n",
        encoding="utf-8",
    )
    (run_paths["environment"] / "runtime.json").write_text(
        json.dumps(
            {
                "python": sys.version,
                "platform": platform.platform(),
                "cwd": str(REPO_ROOT),
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    write_run_status(REPO_ROOT, EXPORT_RUN_ID, stage="export", status="running")

    export_mode = None
    direct_error = None
    merged_error = None

    for loader_name, loader in (("FastVisionModel", FastVisionModel), ("FastLanguageModel", FastLanguageModel)):
        try:
            model, tokenizer = loader.from_pretrained(
                model_name=str(FINAL_MODEL_DIR),
                max_seq_length=4096,
                load_in_4bit=False,
                dtype=torch.bfloat16,
                trust_remote_code=True,
            )
            model.save_pretrained_gguf(
                str(GGUF_DIR),
                tokenizer,
                quantization_method=GGUF_QUANTIZATION_METHODS,
            )
            export_mode = f"{loader_name}.save_pretrained_gguf"
            break
        except Exception as exc:
            direct_error = f"{loader_name}: {exc}"

    if export_mode is None:
        try:
            merged_model = AutoPeftModelForCausalLM.from_pretrained(str(FINAL_MODEL_DIR), trust_remote_code=True)
            merged_model = merged_model.merge_and_unload()
            merged_model.save_pretrained(str(MERGED_DIR))
            export_mode = "merged_16bit_only"
        except Exception as merged_exc:
            merged_error = str(merged_exc)

    gguf_files = detect_gguf_files(GGUF_DIR) if GGUF_DIR.exists() else []
    variants = validate_required_variants(gguf_files) if gguf_files else {
        "has_q8_0": False,
        "has_q4_candidate": False,
    }
    GGUF_EXPORT_REPORT = {
        "run_id": EXPORT_RUN_ID,
        "final_model_dir": str(FINAL_MODEL_DIR),
        "gguf_dir": str(GGUF_DIR),
        "merged_dir": str(MERGED_DIR),
        "export_mode": export_mode,
        "gguf_files": gguf_files,
        "variants": variants,
        "direct_error": direct_error,
        "merged_error": merged_error,
    }

    report_path = run_paths["reports"] / "gguf_export.json"
    report_path.write_text(json.dumps(GGUF_EXPORT_REPORT, indent=2), encoding="utf-8")
    checksum_payload = {
        "reports/gguf_export.json": sha256_file(report_path),
    }
    if GGUF_DIR.exists():
        checksum_payload["gguf_artifacts"] = {
            str(path.relative_to(GGUF_DIR)): sha256_file(path)
            for path in sorted(GGUF_DIR.rglob("*"))
            if path.is_file()
        }
    if MERGED_DIR.exists():
        checksum_payload["merged_artifacts"] = {
            str(path.relative_to(MERGED_DIR)): sha256_file(path)
            for path in sorted(MERGED_DIR.rglob("*"))
            if path.is_file()
        }
    (run_paths["checksums"] / "artifacts.json").write_text(
        json.dumps(checksum_payload, indent=2),
        encoding="utf-8",
    )

    if not (variants["has_q8_0"] and variants["has_q4_candidate"]):
        write_run_status(
            REPO_ROOT,
            EXPORT_RUN_ID,
            stage="export",
            status="failed",
            details=GGUF_EXPORT_REPORT,
        )
        raise RuntimeError(
            "GGUF export did not produce the required q8_0 and q4 variants. "
            "The report remains under workspace/runs/<run_id>/reports/gguf_export.json."
        )

    write_run_status(
        REPO_ROOT,
        EXPORT_RUN_ID,
        stage="export",
        status="completed",
        details=GGUF_EXPORT_REPORT,
    )
    write_run_summary(
        REPO_ROOT,
        EXPORT_RUN_ID,
        "# GGUF Export Summary\n\n```json\n" + json.dumps(GGUF_EXPORT_REPORT, indent=2) + "\n```",
    )
    print(json.dumps(GGUF_EXPORT_REPORT, indent=2))
else:
    print(json.dumps(GGUF_EXPORT_REPORT, indent=2))


In [ ]:
from __future__ import annotations

import json
import platform
import sys

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from lumis1.export_smoke import run_export_smoke
from lumis1.hashing import sha256_file
from lumis1.run_evidence import (
    assess_eval_export_status,
    create_run_evidence_tree,
    write_run_status,
    write_run_summary,
)

REPORTS = REPO_ROOT / "workspace" / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

def render_prompt(tokenizer: AutoTokenizer, prompt: str) -> str:
    rendered = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    if "<think>" in rendered or "</think>" in rendered:
        raise RuntimeError("Rendered eval prompt still contains thinking markers")
    return rendered

identity_prompts = [
    "What is your name?",
    "Who created you?",
]
neutral_prompts = [
    "Summarize this paragraph in one sentence.",
    "What is the weather process for rain formation?",
    "Rewrite this sentence in simpler English.",
]
no_image_prompts = [
    "Describe the image.",
    "What objects are visible here?",
]

final_model_dir = DPO_OUTPUT_DIR if RUN_DPO else SFT_OUTPUT_DIR
results = {
    "run_eval": RUN_EVAL,
    "run_export": RUN_EXPORT,
    "run_id": EVAL_RUN_ID,
    "final_model_dir": str(final_model_dir),
    "checks": {
        "identity_correctness": {"status": "skipped"},
        "self_branding_rate": {"status": "skipped"},
        "vision_hallucination_on_no_image": {"status": "skipped"},
        "multimodal_correctness": {"status": "skipped"},
    },
}

if RUN_EVAL or RUN_EXPORT:
    run_paths = create_run_evidence_tree(REPO_ROOT, EVAL_RUN_ID)
    (run_paths["config_snapshot"] / "eval_export_inputs.json").write_text(
        json.dumps(
            {
                "model_dir": str(final_model_dir),
                "gguf_dir": str(GGUF_DIR),
                "run_eval": RUN_EVAL,
                "run_export": RUN_EXPORT,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    (run_paths["commands"] / "notebook_90_eval_invocation.txt").write_text(
        "Sequential Colab notebook eval/export smoke execution\n"
        f"run_eval={RUN_EVAL}\n"
        f"run_export={RUN_EXPORT}\n"
        f"model_dir={final_model_dir}\n"
        f"gguf_dir={GGUF_DIR}\n",
        encoding="utf-8",
    )
    (run_paths["environment"] / "runtime.json").write_text(
        json.dumps(
            {
                "python": sys.version,
                "platform": platform.platform(),
                "cwd": str(REPO_ROOT),
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    write_run_status(REPO_ROOT, EVAL_RUN_ID, stage="eval_export", status="running")

    if RUN_EVAL:
        if not final_model_dir.exists():
            raise FileNotFoundError(f"final model dir not found: {final_model_dir}")

        tokenizer = AutoTokenizer.from_pretrained(final_model_dir, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(final_model_dir, torch_dtype=torch.bfloat16, trust_remote_code=True)
        model.eval()
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"
        torch.manual_seed(EVAL_SEED)

        def generate(prompt: str, max_new_tokens: int = 128) -> str:
            rendered_prompt = render_prompt(tokenizer, prompt)
            toks = tokenizer(rendered_prompt, return_tensors="pt")
            toks = {key: value.to(model.device) for key, value in toks.items()}
            with torch.no_grad():
                out = model.generate(
                    **toks,
                    max_new_tokens=max_new_tokens,
                    do_sample=EVAL_DO_SAMPLE,
                    temperature=EVAL_TEMPERATURE if EVAL_DO_SAMPLE else None,
                    top_p=EVAL_TOP_P if EVAL_DO_SAMPLE else None,
                    top_k=EVAL_TOP_K if EVAL_DO_SAMPLE else None,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            generated = out[0][toks["input_ids"].shape[-1]:]
            return tokenizer.decode(generated, skip_special_tokens=True).strip()

        identity_outputs = [generate(prompt) for prompt in identity_prompts]
        identity_ok = all(
            ("Lumis-1" in text or "Lumis" in text) and ("Eptesicus" in text)
            for text in identity_outputs
        )
        results["checks"]["identity_correctness"] = {
            "status": "pass" if identity_ok else "fail",
            "outputs": identity_outputs,
        }

        neutral_outputs = [generate(prompt) for prompt in neutral_prompts]
        brand_hits = sum(1 for text in neutral_outputs if ("Lumis" in text or "Eptesicus" in text))
        results["checks"]["self_branding_rate"] = {
            "status": "pass",
            "rate": brand_hits / max(len(neutral_outputs), 1),
            "outputs": neutral_outputs,
        }

        no_image_outputs = [generate(prompt) for prompt in no_image_prompts]
        hallucination_hits = sum(
            1
            for text in no_image_outputs
            if "image" not in text.lower() and "can't" not in text.lower()
        )
        results["checks"]["vision_hallucination_on_no_image"] = {
            "status": "pass" if hallucination_hits == 0 else "warn",
            "hallucination_hits": hallucination_hits,
            "outputs": no_image_outputs,
        }

        eval_mm_path = REPORTS / "eval_multimodal_prompts.json"
        if eval_mm_path.exists():
            mm_items = json.loads(eval_mm_path.read_text(encoding="utf-8"))
            results["checks"]["multimodal_correctness"] = {
                "status": "manual_review_required",
                "items_loaded": len(mm_items),
            }
        else:
            results["checks"]["multimodal_correctness"] = {
                "status": "skipped",
                "reason": "workspace/reports/eval_multimodal_prompts.json not provided",
            }

    parity_pairs_path = REPORTS / "template_parity_pairs.json"
    parity_pairs = json.loads(parity_pairs_path.read_text(encoding="utf-8")) if parity_pairs_path.exists() else []
    if RUN_EXPORT and GGUF_DIR.exists() and parity_pairs:
        smoke = run_export_smoke(GGUF_DIR, parity_pairs)
        results["export_smoke"] = {
            "status": "pass" if smoke["ok"] else "fail",
            **smoke,
        }
    elif RUN_EXPORT and GGUF_DIR.exists():
        results["export_smoke"] = {
            "status": "partial",
            "reason": "GGUF files exist but workspace/reports/template_parity_pairs.json is missing",
        }
    else:
        results["export_smoke"] = {
            "status": "skipped",
            "reason": "GGUF export directory not found or export disabled",
        }

    out = REPORTS / "export_smoke.json"
    out.write_text(json.dumps(results, indent=2), encoding="utf-8")
    (run_paths["reports"] / "export_smoke.json").write_text(
        json.dumps(results, indent=2),
        encoding="utf-8",
    )
    checksum_payload = {
        "export_smoke_report": sha256_file(out),
    }
    if GGUF_DIR.exists():
        checksum_payload["gguf_artifacts"] = {
            str(path.relative_to(GGUF_DIR)): sha256_file(path)
            for path in sorted(GGUF_DIR.rglob("*"))
            if path.is_file()
        }
    (run_paths["checksums"] / "artifacts.json").write_text(
        json.dumps(checksum_payload, indent=2),
        encoding="utf-8",
    )
    assessment = assess_eval_export_status(
        results=results,
        run_eval=RUN_EVAL,
        run_export=RUN_EXPORT,
        run_paths=run_paths,
    )
    results["run_status_assessment"] = assessment
    write_run_status(
        REPO_ROOT,
        EVAL_RUN_ID,
        stage="eval_export",
        status=assessment["status"],
        details=results,
    )
    write_run_summary(
        REPO_ROOT,
        EVAL_RUN_ID,
        "# Eval / Export Summary\n\n```json\n" + json.dumps(results, indent=2) + "\n```",
    )
    print(json.dumps(results, indent=2))
else:
    print(json.dumps(results, indent=2))


In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

if not EXPORT_PERSIST_ROOT.exists():
    EXPORT_PERSIST_ROOT.mkdir(parents=True, exist_ok=True)

export_root = EXPORT_PERSIST_ROOT / PIPELINE_PREFIX
if export_root.exists():
    shutil.rmtree(export_root)
export_root.mkdir(parents=True, exist_ok=True)

copy_targets = {
    "reports": REPO_ROOT / "workspace" / "reports",
    "final": REPO_ROOT / "workspace" / "final",
    "runs": REPO_ROOT / "workspace" / "runs",
}
copied = {}
for name, source in copy_targets.items():
    if source.exists():
        destination = export_root / name
        shutil.copytree(source, destination, dirs_exist_ok=True)
        copied[name] = str(destination)

gguf_candidates = []
if GGUF_DIR.exists():
    gguf_candidates = sorted(str(path) for path in GGUF_DIR.glob("*.gguf"))

summary = {
    "export_root": str(export_root),
    "copied": copied,
    "gguf_candidates": gguf_candidates,
}
print(json.dumps(summary, indent=2))

if DOWNLOAD_GGUF_TO_BROWSER and gguf_candidates and "google.colab" in sys.modules:
    from google.colab import files  # type: ignore

    chosen = next(
        (path for path in gguf_candidates if DOWNLOAD_GGUF_VARIANT_TOKEN.lower() in Path(path).name.lower()),
        gguf_candidates[0],
    )
    print(f"downloading: {chosen}")
    files.download(chosen)
